In [ ]:
pip install nbstripout

In [ ]:
!nbstripout Copy_of_dicta_and_NER_task.ipynb

In [ ]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

import re
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    pipeline
)
from sklearn.metrics import f1_score, accuracy_score, classification_report

ner_model_name = "dicta-il/dictabert-ner"
ner = pipeline(
    task="token-classification",
    model=ner_model_name,
    tokenizer=ner_model_name,
    aggregation_strategy="simple",  # merges sub-tokens into full spans
    device=0 if torch.cuda.is_available() else -1
)



In [ ]:
def normalize_ner_label(lbl: str) -> str:
    # pipeline may output "PER", "ORG" or "B-PER"/"I-PER" depending on model
    lbl = lbl.replace("B-", "").replace("I-", "")
    return lbl.upper()

def inject_ner_markers(text: str, ner_spans, min_score=0.50):
    """
    text: original sentence
    ner_spans: output of pipeline("token-classification") with aggregation_strategy="simple"
    returns: text with markers like <PER>... </PER>
    """
    if not text or not ner_spans:
        return text

    # Keep only confident spans
    spans = []
    for s in ner_spans:
        if s.get("score", 1.0) >= min_score and "start" in s and "end" in s:
            label = normalize_ner_label(s.get("entity_group", s.get("entity", "ENT")))
            spans.append((s["start"], s["end"], label))

    if not spans:
        return text

    # Remove overlaps (keep longer/high confidence first)
    spans.sort(key=lambda x: (x[0], -(x[1]-x[0])))
    filtered = []
    last_end = -1
    for start, end, label in spans:
        if start >= last_end:
            filtered.append((start, end, label))
            last_end = end

    # Insert tags from end → start so indices stay valid
    out = text
    for start, end, label in sorted(filtered, key=lambda x: x[0], reverse=True):
        out = out[:end] + f"</{label}>" + out[end:]
        out = out[:start] + f"<{label}>" + out[start:]
    return out

def mark_pair(example):
    s1 = example["translation1"]
    s2 = example["translation2"]

    spans1 = ner(s1) if s1 else []
    spans2 = ner(s2) if s2 else []

    example["marked1"] = inject_ner_markers(s1, spans1)
    example["marked2"] = inject_ner_markers(s2, spans2)
    return example



In [ ]:
data_files = {
    "train": "/content/drive/MyDrive/nlp_pro2/data_sets/use_part/train.jsonl",
    "validation": "/content/drive/MyDrive/nlp_pro2/data_sets/use_part/val.jsonl",
    "test": "/content/drive/MyDrive/nlp_pro2/data_sets/use_part/test.jsonl",
}
ds = load_dataset("json", data_files=data_files)

# ⚠️ Start small for debugging (optional)
# ds["train"] = ds["train"].select(range(5000))
# ds["validation"] = ds["validation"].select(range(1000))
# ds["test"] = ds["test"].select(range(1000))

ds_marked = ds.map(mark_pair)


In [ ]:
model_name = "dicta-il/dictabert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Add tag tokens (you can add more if your NER uses more labels)
special_tags = ["<PER>", "</PER>", "<ORG>", "</ORG>", "<LOC>", "</LOC>", "<MISC>", "</MISC>"]
tokenizer.add_special_tokens({"additional_special_tokens": special_tags})

label_list = ["entailment", "contradiction", "neutral"]
label2id = {l: i for i, l in enumerate(label_list)}

def tokenize_nli_marked(example):
    tok = tokenizer(
        example["marked1"],
        example["marked2"],
        truncation=True,
        max_length=128,
    )
    tok["labels"] = label2id[example["label"]]
    return tok

ds_tok = ds_marked.map(tokenize_nli_marked, remove_columns=ds_marked["train"].column_names)


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score, accuracy_score, classification_report

# -----------------------
# A) Model Definition
# -----------------------
class DictaBERTNLI(nn.Module):
    def __init__(self, model_name: str, num_labels: int = 3, dropout_p: float = 0.1):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout_p)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        # Using the [CLS] token representation for classification
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_output))

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)

        return {"loss": loss, "logits": logits}

# -----------------------
# B) Custom Trainer
# -----------------------
class NLITRainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

# -----------------------
# C) Data & Tokenization - גרסה מתוקנת
# -----------------------
def preprocess_function(example):
    # טוקניזציה של זוג המשפטים
    result = tokenizer(
        example["translation1"],
        example["translation2"],
        truncation=True,
        max_length=128,
        padding=False # ה-DataCollator יטפל בזה בבאטץ'
    )
    result["labels"] = label2id[example["label"]]
    return result

# השורה הקריטית: remove_columns מסירה את כל מה שלא חזר מהפונקציה preprocess_function
ds_tok = ds.map(
    preprocess_function,
    batched=False,
    remove_columns=ds["train"].column_names # מסיר את translation1, translation2, genre וכו'
)

# בדיקה שהכל תקין - אמור להדפיס רק dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
print("Columns in processed dataset:", ds_tok["train"].features.keys())


# -----------------------
# D) Metrics
# -----------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
    }

# -----------------------
# E) Training Execution
# -----------------------
model = DictaBERTNLI(model_name, num_labels=3)

# Resize embeddings in case special tokens were added to tokenizer
model.encoder.resize_token_embeddings(len(tokenizer))

args = TrainingArguments(
    output_dir="dictabert_nli_final_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=10,             # Set high, Early Stopping will handle the rest
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,     # Necessary for Early Stopping
    metric_for_best_model="f1",      # Monitor Macro-F1 for best performance
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    remove_unused_columns=False,     # Essential for custom model classes
    report_to="none",
)

trainer = NLITRainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Stops if no improvement for 3 epochs
)

# Start Training
trainer.train()

# -----------------------
# F) Save & Evaluate
# -----------------------
save_path = "/content/drive/MyDrive/nlp_pro2/save/dictabert_ner_nli_best"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved successfully to {save_path}")

print("\nFINAL TEST RESULTS:")
test_out = trainer.predict(ds_tok["test"])
test_preds = np.argmax(test_out.predictions, axis=-1)
print(classification_report(ds_tok["test"]["labels"], test_preds, target_names=label_list))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

# קבלת התחזיות מהטסט
test_out = trainer.predict(ds_tok["test"])
test_preds = np.argmax(test_out.predictions, axis=-1)
test_labels = ds_tok["test"]["labels"]

# חישוב המטריצה
cm = confusion_matrix(test_labels, test_preds)

print("Confusion Matrix (Raw):")
print(cm)

In [ ]:
def plot_confusion_matrix(cm, labels):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.title('Confusion Matrix - DictaBERT NLI')
    plt.show()

# קריאה לפונקציה עם הלייבלים שלך
plot_confusion_matrix(cm, label_list)

In [ ]:
# 1. קבלת התוצאות מה-Predict
test_out = trainer.predict(ds_tok["test"])
test_preds = np.argmax(test_out.predictions, axis=-1)
test_labels = np.array(ds_tok["test"]["labels"]) # המרת הלייבלים האמיתיים למערך

# 2. חישוב כמות הטעויות
total_mistakes = (test_preds != test_labels).sum()
total_examples = len(test_labels)
accuracy_percent = (1 - total_mistakes / total_examples) * 100

# 3. הדפסת התוצאות
print("-" * 30)
print(f"TOTAL MISTAKES: {total_mistakes} out of {total_examples}")
print(f"ERROR RATE: {(total_mistakes / total_examples):.2%}")
print(f"TOTAL CORRECT: {total_examples - total_mistakes}")
print(f"ACCURACY: {accuracy_percent:.2f}%")
print("-" * 30)

In [ ]:
# יצירת DataFrame לניתוח מהיר
test_df = pd.DataFrame({
    'true': test_labels,
    'pred': test_preds,
    'genre': ds["test"]["genre"]
})

# סינון רק של הטעויות
mistakes_df = test_df[test_df['true'] != test_df['pred']]

print("\nMistakes per Genre:")
print(mistakes_df['genre'].value_counts())

Now to check the type of error

In [ ]:
import torch.nn.functional as F

# 1. קבלת התחזיות וההסתברויות (Softmax)
test_out = trainer.predict(ds_tok["test"])
logits = torch.from_numpy(test_out.predictions)
probs = F.softmax(logits, dim=-1).numpy()
test_preds = np.argmax(probs, axis=-1)
test_labels = ds_tok["test"]["labels"]

# 2. חישוב הביטחון (Confidence) וההפרש בין המקום הראשון לשני (Margin)
pred_conf = np.max(probs, axis=-1)
sorted_probs = np.sort(probs, axis=-1)
margin = sorted_probs[:, -1] - sorted_probs[:, -2]

# 3. בניית ה-DataFrame
# נשלוף את הטקסט המקורי מה-ds["test"] הלא-מעובד
err_df = pd.DataFrame({
    "t1": ds["test"]["translation1"],
    "t2": ds["test"]["translation2"],
    "true": [label_list[i] for i in test_labels],
    "pred": [label_list[i] for i in test_preds],
    "genre": ds["test"]["genre"],
    "pred_conf": pred_conf,
    "margin": margin
})

# נשמור רק את השורות שבהן המודל טעה
err_df = err_df[err_df["true"] != err_df["pred"]].copy()

In [ ]:
import re

NEG_RE = re.compile(r"\b(לא|אין|בלי|אף|מעולם)\b")
TEMP_RE = re.compile(r"\b(אתמול|מחר|היום|השבוע|שנה|שנת|חודש|יום|לילה|בוקר|ערב|שני|שלישי|רביעי|חמישי|שישי|שבת|סוף\s*השבוע)\b")
NUM_RE  = re.compile(r"\b(\d+|אחד|אחת|שניים|שתיים|שלושה|שלוש|ארבע|חמש|שש|שבע|שמונה|תשע|עשר|מאה|אלף|מיליון|אחוז)\b")
PRON_RE = re.compile(r"\b(הוא|היא|הם|הן|אותו|אותה|להם|להן|שלו|שלה|שלהם|שלהן)\b")

def tokenize_set(s):
    return set(re.findall(r"[\u0590-\u05FF]{2,}", s))

def jaccard(a, b):
    if not a or not b: return 0.0
    return len(a & b) / len(a | b)

def detect_patterns(t1, t2, pred_conf=None, margin=None):
    tags = []
    # Negation flip
    if bool(NEG_RE.search(t1)) ^ bool(NEG_RE.search(t2)):
        tags.append("Negation flip")
    # Temporal / Numbers / Pronouns
    if TEMP_RE.search(t1) or TEMP_RE.search(t2): tags.append("Temporal reasoning")
    if NUM_RE.search(t1) or NUM_RE.search(t2): tags.append("Quantifiers / numbers")
    if PRON_RE.search(t1) or PRON_RE.search(t2): tags.append("Coreference / pronouns")
    # Lexical overlap trap
    if jaccard(tokenize_set(t1), tokenize_set(t2)) >= 0.35:
        tags.append("Lexical overlap trap")
    # Ambiguity
    if (pred_conf is not None and pred_conf < 0.45) or (margin is not None and margin < 0.10):
        tags.append("Ambiguity")
    return tags

In [ ]:
# החלת הפונקציה
err_df["tags"] = err_df.apply(
    lambda r: detect_patterns(r["t1"], r["t2"], r["pred_conf"], r["margin"]), axis=1
)

# ניתוח סטטיסטי של הטעויות
exploded = err_df.explode("tags")
tag_counts = exploded["tags"].value_counts()

print("\n--- ERROR ANALYSIS REPORT ---")
print(f"Total mistakes analyzed: {len(err_df)}")
print("\nCommon patterns in mistakes:")
print(tag_counts)

# הדפסת דוגמאות לטעות של "Coreference / pronouns" (הבעיה שרצית לפתור עם NER)
print("\n--- EXAMPLES OF PRONOUN MISTAKES ---")
pronoun_errors = err_df[err_df["tags"].apply(lambda x: "Coreference / pronouns" in x)].head(3)
for _, row in pronoun_errors.iterrows():
    print(f"P: {row['t1']}\nH: {row['t2']}\nTrue: {row['true']} | Pred: {row['pred']}\n")

test with are own examples:

In [ ]:
import torch
import torch.nn.functional as F

def predict_nli(premise, hypothesis, model, tokenizer, label_list):
    # Set model to evaluation mode
    model.eval()

    # 1. Preprocess and Tokenize
    inputs = tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    # Move inputs to the same device as the model (GPU/CPU)
    inputs = {k: v.to(model.encoder.device) for k, v in inputs.items()}

    # 2. Forward Pass
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs["logits"]

    # 3. Get Probabilities and Label
    probs = F.softmax(logits, dim=-1)
    pred_idx = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][pred_idx].item()

    return label_list[pred_idx], confidence

# Example usage:
premise_text = "הוא אוהב אוכל"
hypothesis_text = "הוא שונא אוכל"

label, conf = predict_nli(premise_text, hypothesis_text, model, tokenizer, label_list)
print(f"Prediction: {label} ({conf:.2%} confidence)")

In [ ]:
import pandas as pd

# 1. הפקת המטריצה (cm כבר קיים מהשלב הקודם)
# label_list = ['entailment', 'contradiction', 'neutral']

confusion_data = []
for i, true_label in enumerate(label_list):
    for j, pred_label in enumerate(label_list):
        if i != j:  # אנחנו רוצים רק טעויות (לא האלכסון)
            confusion_data.append({
                'True Label': true_label,
                'Predicted Label': pred_label,
                'Count': cm[i][j]
            })

# 2. המרה ל-DataFrame ומיון מהגבוה לנמוך
confusion_df = pd.DataFrame(confusion_data)
confusion_df = confusion_df.sort_values(by='Count', ascending=False)

print("Top Label Confusions:")
print(confusion_df)

In [ ]:
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

# קבלת התוצאות מהטסט
test_out = trainer.predict(ds_tok["test"])

# הפיכת ה-Logits להסתברויות (Softmax) שסוכמות ל-1
logits = torch.from_numpy(test_out.predictions)
probs = F.softmax(logits, dim=-1).numpy()

# הלייבלים האמיתיים
y_true = ds_tok["test"]["labels"]

In [ ]:
# חישוב Macro ROC-AUC: נותן משקל שווה לכל מחלקה
roc_auc_macro = roc_auc_score(
    y_true,
    probs,
    multi_class='ovr',
    average='macro'
)

# חישוב Weighted ROC-AUC: נותן משקל לפי כמות הדוגמאות בכל מחלקה
roc_auc_weighted = roc_auc_score(
    y_true,
    probs,
    multi_class='ovr',
    average='weighted'
)

print(f"ROC-AUC Macro (OvR): {roc_auc_macro:.4f}")
print(f"ROC-AUC Weighted (OvR): {roc_auc_weighted:.4f}")

different way to test wiyh our examples